# **Install**

In [9]:
!pip install transformers datasets torch seqeval -q

# **Dataset Load and Explore**

In [10]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("medalpaca/medical_meadow_pubmed_causal")
df = pd.DataFrame(dataset['train'])
print(df[['input','output']].head(5).to_string())

                                                                                                                                                                                     input                                          output
0  A McMahon score of at least 6 calculated on admission allows for a more sensitive, specific and timely identification of patients who may benefit from high-volume fluid resuscitation.     This is a directly correlative relationship
1                                                                  The rs7903146 (C/T) polymorphism of the TCF7L2 gene might not be associated with obesity in the Cameroonian population.  This is a conditionally causative relationship
2                                                                                A fifth of patients with a negative CDS of the TAs had signs of vasculitis on the CDS of the FaA, or OcA.  This is a conditionally causative relationship
3                                                           

# **BIO Tags**

In [17]:
import re
from datasets import Dataset, DatasetDict

CAUSAL_TRIGGERS = [
    'causes', 'cause', 'caused by', 'leads to', 'lead to',
    'results in', 'result in', 'produces', 'induces', 'triggers',
    'increases risk of', 'associated with', 'due to',
    'because of', 'is caused by', 'contributes to',
    'increases', 'reduces', 'inhibits', 'promotes', 'affects'
]

def find_cause_effect_by_trigger(sentence):
    sentence_lower = sentence.lower()
    for trigger in sorted(CAUSAL_TRIGGERS, key=len, reverse=True):
        pattern = r'\b' + re.escape(trigger) + r'\b'
        match = re.search(pattern, sentence_lower)
        if match:
            cause  = sentence[:match.start()].strip().rstrip('.,;')
            effect = sentence[match.end():].strip().lstrip('.,; ').rstrip('.')
            if cause and effect:
                return cause, effect, trigger
    return None, None, None

def create_bio_tags(tokens, cause_span, effect_span):
    tags = ['O'] * len(tokens)

    def tag_span(span, b_tag, i_tag):
        if not span:
            return
        # Span tokens
        span_tokens = [t.lower().strip('.,;:') for t in span.split()]
        src_tokens  = [t.lower().strip('.,;:') for t in tokens]
        n = len(span_tokens)
        for i in range(len(src_tokens) - n + 1):
            if src_tokens[i:i+n] == span_tokens:
                tags[i] = b_tag
                for j in range(1, n):
                    tags[i+j] = i_tag
                return

    tag_span(cause_span,  'B-CAUSE',  'I-CAUSE')
    tag_span(effect_span, 'B-EFFECT', 'I-EFFECT')
    return tags

# ---- Convert ----
ner_data = []
skipped  = 0
cause_only = 0

for row in dataset['train']:
    sentence = row['input'].strip()
    cause, effect, trigger = find_cause_effect_by_trigger(sentence)

    if cause and effect:
        tokens = sentence.split()
        tags   = create_bio_tags(tokens, cause, effect)

        has_cause  = any(t in ('B-CAUSE',  'I-CAUSE')  for t in tags)
        has_effect = any(t in ('B-EFFECT', 'I-EFFECT') for t in tags)

        if has_cause and has_effect:
            ner_data.append({
                'tokens':       tokens,
                'ner_tags_str': tags,
                'sentence':     sentence,
                'cause':        cause,
                'effect':       effect,
                'trigger':      trigger
            })
        elif has_cause:
            cause_only += 1
            skipped += 1
        else:
            skipped += 1
    else:
        skipped += 1

print(f"NER samples (cause+effect both): {len(ner_data)}")
print(f"Cause only (dropped)           : {cause_only}")
print(f"Skipped (no trigger found)     : {skipped}")

# ---- Verify top 5 samples ----
print("\n=== Top 5 samples verification ===")
for i, sample in enumerate(ner_data[:5]):
    print(f"\n[{i}] {sample['sentence']}")
    print(f"     CAUSE : {sample['cause']}")
    print(f"     EFFECT: {sample['effect']}")
    print(f"     TRIGGER: {sample['trigger']}")
    print("     TAGS:", list(zip(sample['tokens'], sample['ner_tags_str'])))

NER samples (cause+effect both): 333
Cause only (dropped)           : 1
Skipped (no trigger found)     : 2113

=== Top 5 samples verification ===

[0] The rs7903146 (C/T) polymorphism of the TCF7L2 gene might not be associated with obesity in the Cameroonian population.
     CAUSE : The rs7903146 (C/T) polymorphism of the TCF7L2 gene might not be
     EFFECT: obesity in the Cameroonian population
     TRIGGER: associated with
     TAGS: [('The', 'B-CAUSE'), ('rs7903146', 'I-CAUSE'), ('(C/T)', 'I-CAUSE'), ('polymorphism', 'I-CAUSE'), ('of', 'I-CAUSE'), ('the', 'I-CAUSE'), ('TCF7L2', 'I-CAUSE'), ('gene', 'I-CAUSE'), ('might', 'I-CAUSE'), ('not', 'I-CAUSE'), ('be', 'I-CAUSE'), ('associated', 'O'), ('with', 'O'), ('obesity', 'B-EFFECT'), ('in', 'I-EFFECT'), ('the', 'I-EFFECT'), ('Cameroonian', 'I-EFFECT'), ('population.', 'I-EFFECT')]

[1] The findings of this study substantiate the need for physical exercise to reduce  signs and symptoms associated with CVD risk, even among a young, healt

# **Label Encoding + HuggingFace Dataset**

In [18]:
from datasets import Dataset, DatasetDict

LABEL_LIST = ['O', 'B-CAUSE', 'I-CAUSE', 'B-EFFECT', 'I-EFFECT']
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}

# String tags → integers
for item in ner_data:
    item['ner_tags'] = [LABEL2ID[t] for t in item['ner_tags_str']]

# HuggingFace Dataset
hf_dataset = Dataset.from_list([
    {'tokens': d['tokens'], 'ner_tags': d['ner_tags']}
    for d in ner_data
])

# Train / Val / Test split
split    = hf_dataset.train_test_split(test_size=0.15, seed=42)
val_test = split['test'].train_test_split(test_size=0.5, seed=42)

final_dataset = DatasetDict({
    'train':      split['train'],
    'validation': val_test['train'],
    'test':       val_test['test']
})

print(final_dataset)
print("\nLabel mapping:", LABEL2ID)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 283
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 25
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 25
    })
})

Label mapping: {'O': 0, 'B-CAUSE': 1, 'I-CAUSE': 2, 'B-EFFECT': 3, 'I-EFFECT': 4}


# **Tokenize (subword alignment)**

In [19]:
from transformers import AutoTokenizer

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        padding='max_length',
        max_length=128,
        is_split_into_words=True   #  tokens list
    )
    all_labels = []
    for i, word_ids in enumerate(tokenized.word_ids(batch_index=i)
                                  for i in range(len(examples['tokens']))):
        labels     = []
        prev_word  = None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)          # [CLS], [SEP], padding
            elif wid != prev_word:
                labels.append(examples['ner_tags'][i][wid])
            else:

                orig = examples['ner_tags'][i][wid]
                # B-CAUSE → I-CAUSE, B-EFFECT → I-EFFECT
                if orig == LABEL2ID['B-CAUSE']:
                    labels.append(LABEL2ID['I-CAUSE'])
                elif orig == LABEL2ID['B-EFFECT']:
                    labels.append(LABEL2ID['I-EFFECT'])
                else:
                    labels.append(orig)
            prev_word = wid
        all_labels.append(labels)

    tokenized['labels'] = all_labels
    return tokenized

tokenized_dataset = final_dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=['tokens', 'ner_tags']
)
print("Done!", tokenized_dataset)

Map:   0%|          | 0/283 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Done! DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 283
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 25
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 25
    })
})


# **Fine-Tune**

In [20]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from seqeval.metrics import f1_score, classification_report
import numpy as np, torch

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    true_preds  = [[ID2LABEL[p] for p, l in zip(pred, lab) if l != -100]
                    for pred, lab in zip(preds, labels)]
    true_labels = [[ID2LABEL[l] for l in lab if l != -100]
                    for lab in labels]

    return {
        "f1": f1_score(true_labels, true_preds),
    }

training_args = TrainingArguments(
    output_dir="./causal_ner_model",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="epoch",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
trainer.save_model("./causal_ner_model")
tokenizer.save_pretrained("./causal_ner_model")
print("Model saved!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

Epoch,Training Loss,Validation Loss,F1
1,1.588905,1.307775,0.020513
2,1.050374,0.812140,0.060000
3,0.567600,0.505426,0.581197
4,0.148731,0.341711,0.796296
5,0.037417,0.324528,0.830189
6,0.017639,0.335332,0.893204


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


# **Extraction**

In [21]:
from transformers import pipeline

ner_pipe = pipeline(
    "ner",
    model="./causal_ner_model",
    tokenizer="./causal_ner_model",
    aggregation_strategy="simple",   # words merge ho jayein
    device=0 if torch.cuda.is_available() else -1
)

def extract_cause_effect_from_sentence(sentence):
    entities = ner_pipe(sentence)
    cause, effect = [], []
    for ent in entities:
        if 'CAUSE'  in ent['entity_group']: cause.append(ent['word'])
        if 'EFFECT' in ent['entity_group']: effect.append(ent['word'])
    return {
        "sentence": sentence,
        "cause":    ' '.join(cause)  if cause  else "not found",
        "effect":   ' '.join(effect) if effect else "not found"
    }

# Test
sentences = [
    "Cigarette smoking causes lung cancer.",
    "Diabetes leads to increased risk of cardiovascular disease.",
    "Aspirin reduces inflammation by inhibiting COX enzymes.",
    "Obesity increases the risk of developing type 2 diabetes.",
    "COVID-19 infection causes cytokine storm in severe cases.",
]

print("=" * 55)
print(f"{'SENTENCE':<40} | CAUSE → EFFECT")
print("=" * 55)
for s in sentences:
    result = extract_cause_effect_from_sentence(s)
    print(f"\nSentence : {result['sentence']}")
    print(f"  CAUSE  : {result['cause']}")
    print(f"  EFFECT : {result['effect']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SENTENCE                                 | CAUSE → EFFECT

Sentence : Cigarette smoking causes lung cancer.
  CAUSE  : Cigarette smoking
  EFFECT : lung cancer.

Sentence : Diabetes leads to increased risk of cardiovascular disease.
  CAUSE  : Diabetes
  EFFECT : increased risk of cardiovascular disease.

Sentence : Aspirin reduces inflammation by inhibiting COX enzymes.
  CAUSE  : Aspirin
  EFFECT : inflammation by inhibiting COX enzymes.

Sentence : Obesity increases the risk of developing type 2 diabetes.
  CAUSE  : Obesity
  EFFECT : the risk of developing type 2 diabetes.

Sentence : COVID-19 infection causes cytokine storm in severe cases.
  CAUSE  : COVID - 19 infection
  EFFECT : cytokine storm in severe cases.
